# NLP Text Preprocessing 

## The Roadmap

We'll go through preprocessing in the same order a real pipeline uses it:

| # | Stage | What it does |
|---|-------|---------------|
| 1 | Setup & Sample Data | Get some deliberately messy text to practice on |
| 2 | Cleaning (Noise Removal) | Strip HTML, URLs, emojis, punctuation, slang, typos |
| 3 | Tokenization | Break text into words / subwords the model can consume |
| 4 | Stopword Removal | Drop low-information filler words ("the", "is", "a") |
| 5 | Stemming vs Lemmatization | Reduce words to their root form |
| 6 | POS Tagging & NER | Extract grammatical & entity signals (used in some pipelines) |
| 7 | Reusable Pipeline Class | Package everything into one copy-paste `TextPreprocessor` |
| 8 | Apply to Real Data | Run the pipeline on actual IMDB movie reviews |
| 9 | Text → Numbers | Bag-of-Words & TF-IDF (the bridge to modeling) |
| 10 | Best Practices & Pitfalls | What big tech teams actually do differently, and why |
| 11 | Cheat Sheet | Every snippet in one place for fast copy-paste |

> 💡 **Golden rule you'll see repeated throughout:** *there is no single "correct" preprocessing recipe.* The right steps depend entirely on your downstream task and model. We'll flag this every time it matters.


## 1. Setup & Sample Data

First, the libraries. Everything here is either in the Python standard library or a well-known, stable NLP package — the same tools used in production pipelines at most companies.

Run this cell once. If a library is missing, `pip install` it (commands are commented next to each import).


In [17]:
%pip install pandas emoji contractions unidecode pyspellchecker nltk spacy scikit-learn

   ---------------------------------------- 0.0/608.4 kB ? eta -:--:--
   ---------------------------------------- 608.4/608.4 kB 7.5 MB/s  0:00:00
   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   ----- ---------------------------------- 1.0/7.2 MB 4.4 MB/s eta 0:00:02
   -------- ------------------------------- 1.6/7.2 MB 3.7 MB/s eta 0:00:02
   ------------- -------------------------- 2.4/7.2 MB 3.6 MB/s eta 0:00:02
   --------------- ------------------------ 2.9/7.2 MB 3.5 MB/s eta 0:00:02
   ------------------ --------------------- 3.4/7.2 MB 3.3 MB/s eta 0:00:02
   --------------------- ------------------ 3.9/7.2 MB 3.2 MB/s eta 0:00:02
   ------------------------ --------------- 4.5/7.2 MB 3.1 MB/s eta 0:00:01
   ---------------------------- ----------- 5.2/7.2 MB 3.1 MB/s eta 0:00:01
   ------------------------------- -------- 5.8/7.2 MB 3.1 MB/s eta 0:00:01
   ---------------------------------- ----- 6.3/7.2 MB 3.1 MB/s eta 0:00:01
   --------------------

In [19]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     --- ------------------------------------ 1.0/12.8 MB 10.4 MB/s eta 0:00:02
     ---- ----------------------------------- 1.6/12.8 MB 4.4 MB/s eta 0:00:03
     ------- -------------------------------- 2.4/12.8 MB 4.7 MB/s eta 0:00:03
     --------- ------------------------------ 3.1/12.8 MB 4.1 MB/s eta 0:00:03
     ----------- ---------------------------- 3.7/12.8 MB 3.9 MB/s eta 0:00:03
     ------------- -------------------------- 4.2/12.8 MB 3.8 MB/s eta 0:00:03
     --------------- ------------------------ 5.0/12.8 MB 3.6 MB/s eta 0:00:03
     ----------------- ---------------------- 5.5/12.8 MB 3.6 MB/s eta 0:00:03
     ------------------ --------------------- 6.0/12.8 MB 3.5 MB/s eta 0:00:02
     --------------------- ------------------ 6.8/12.8 MB 3.5 MB/s eta 0:00:02
     ---------------------- ----------------- 7.3/12.8 MB 3.4 MB/s eta 0:00:02
     ------------------------- -------------- 8.1/12.8 MB 

In [20]:
# ---- Standard library ----
import re
import string
import unicodedata

# ---- Data handling ----
import pandas as pd

# ---- Cleaning helpers ----
import emoji                      # pip install emoji
import contractions                # pip install contractions
from unidecode import unidecode    # pip install unidecode
from spellchecker import SpellChecker  # pip install pyspellchecker

# ---- Classic NLP toolkit ----
import nltk                        # pip install nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer
from nltk import pos_tag

# One-time downloads (safe to re-run — it skips what's already downloaded)
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet",
            "omw-1.4", "averaged_perceptron_tagger",
            "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)

# ---- Industry-standard NLP pipeline ----
import spacy                       # pip install spacy
# python -m spacy download en_core_web_sm      <-- run once in a terminal
nlp = spacy.load("en_core_web_sm")

# ---- Vectorization (bridge to modeling, Section 9) ----
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

print("✅ All libraries loaded successfully.")


✅ All libraries loaded successfully.


### 1.1 A deliberately messy sample sentence

Real text is never this bad in one sentence — but packing every kind of noise into one example makes it easy to *see* what each cleaning step does. Keep this cell around; it's your test bench for trying new techniques.


In [2]:
sample_text = (
    "OMG!!! I just watched <b>Inception</b> 😍😍 at https://cinema.example.com/movie/123 "
    "& it's soooo gr8, def a masterpiece... didn't expect that twist ¯\\_(ツ)_/¯ "
    "café scene was epic too. rating: 9/10 #mindblown @director_nolan"
)
print(sample_text)


OMG!!! I just watched <b>Inception</b> 😍😍 at https://cinema.example.com/movie/123 & it's soooo gr8, def a masterpiece... didn't expect that twist ¯\_(ツ)_/¯ café scene was epic too. rating: 9/10 #mindblown @director_nolan


### 1.2 A small real dataset to preprocess later (Section 8)

This continues from your **Notebook 01 (Data Acquisition)**, which downloads the full 50K-row IMDB Movie Reviews dataset via `kagglehub`. To keep *this* notebook runnable on its own (no Kaggle credentials needed), we recreate a small sample with the same two columns (`review`, `sentiment`).

**In your real project:** just delete this cell and use the `df` you already loaded in Notebook 01 — every function below works identically on the full 50K rows.


In [3]:
df = pd.DataFrame({
    "review": [
        "One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked.",
        "A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion.",
        "I thought this was a wonderful way to spend time on a too hot summer weekend, sitting in the air conditioned theater.",
        "Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his parents are fighting all the time.",
        "Petter Mattei's \"Love in the Time of Money\" is a visually stunning film to watch. Mr. Mattei offers us a vivid portrait about human relations.",
        "This movie is soooo bad, I can't believe I wasted 2 hrs on it!! definitely NOT recommended :(",
        "Absolutely brilliant!! Best film I've seen in 2024. 10/10 would watch again <3",
    ],
    "sentiment": ["positive", "positive", "positive", "negative", "positive", "negative", "positive"],
})
df.head()


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


## 2. Cleaning Techniques (Noise Removal)

This is the "beginner layer" — but it's also exactly what production pipelines start with. Each technique below is a **small, independent function**. This is deliberate: in a real pipeline, you rarely apply *every* technique — you pick the subset your task needs (more on this in Section 10). Independent functions mean you can mix and match freely.

We apply each one to `sample_text` so you can see the exact before → after effect.


### 2.1 Lowercasing

The simplest and most common step. `"Movie"`, `"MOVIE"`, and `"movie"` all become `"movie"` — the model doesn't need to learn 3 separate words for one concept.

⚠️ **Task-dependent caveat:** skip this for tasks where case carries meaning — e.g. Named Entity Recognition (capital letters help identify proper nouns) or when using a *cased* transformer model (BERT-cased, GPT models — these already handle case intelligently).


In [4]:
def to_lowercase(text: str) -> str:
    return text.lower()

print(to_lowercase(sample_text))


omg!!! i just watched <b>inception</b> 😍😍 at https://cinema.example.com/movie/123 & it's soooo gr8, def a masterpiece... didn't expect that twist ¯\_(ツ)_/¯ café scene was epic too. rating: 9/10 #mindblown @director_nolan


### 2.2 Removing HTML Tags

Text scraped from the web (like IMDB reviews!) is full of leftover tags such as `<br />`. A simple regex strips anything between `<` and `>`.


In [5]:
def remove_html_tags(text: str) -> str:
    pattern = re.compile(r"<.*?>")
    return pattern.sub("", text)

print(remove_html_tags("A wonderful little production. <br /><br />The filming technique is great."))


A wonderful little production. The filming technique is great.


### 2.3 Removing URLs

URLs are almost never useful signal for text-understanding tasks and just add noise/vocabulary bloat.


In [6]:
def remove_urls(text: str) -> str:
    pattern = re.compile(r"https?://\S+|www\.\S+")
    return pattern.sub("", text)

print(remove_urls(sample_text))


OMG!!! I just watched <b>Inception</b> 😍😍 at  & it's soooo gr8, def a masterpiece... didn't expect that twist ¯\_(ツ)_/¯ café scene was epic too. rating: 9/10 #mindblown @director_nolan


### 2.4 Removing Mentions & Hashtags (social media text)

Common for tweets/comments. Usually you either **delete** `@username` / `#tag` entirely, or **strip just the symbol** and keep the word (hashtags often contain real content, e.g. `#mindblown` → `mindblown`).


In [7]:
def remove_mentions(text: str) -> str:
    return re.sub(r"@\w+", "", text)

def strip_hashtag_symbol(text: str) -> str:
    # keeps the word, removes only the '#'
    return re.sub(r"#(\w+)", r"\1", text)

print(remove_mentions(sample_text))
print(strip_hashtag_symbol(sample_text))


OMG!!! I just watched <b>Inception</b> 😍😍 at https://cinema.example.com/movie/123 & it's soooo gr8, def a masterpiece... didn't expect that twist ¯\_(ツ)_/¯ café scene was epic too. rating: 9/10 #mindblown 
OMG!!! I just watched <b>Inception</b> 😍😍 at https://cinema.example.com/movie/123 & it's soooo gr8, def a masterpiece... didn't expect that twist ¯\_(ツ)_/¯ café scene was epic too. rating: 9/10 mindblown @director_nolan


### 2.5 Handling Accented / Unicode Characters

Words like `café`, `naïve`, or `résumé` should usually be normalized to their plain-ASCII form (`cafe`, `naive`, `resume`) so your vocabulary doesn't split one word into multiple variants.


In [8]:
def remove_accented_chars(text: str) -> str:
    return unidecode(text)

print(remove_accented_chars("café naïve résumé"))


cafe naive resume


### 2.6 Handling Emojis

Two valid strategies — pick based on your task:

1. **Remove** emojis entirely (simplest, fine for topic classification, search indexing).
2. **Convert to text** (`😍` → `:heart_eyes:` → `heart eyes`) — often **better for sentiment analysis**, since emojis carry real emotional signal you don't want to throw away!


In [9]:
def remove_emojis(text: str) -> str:
    return emoji.replace_emoji(text, replace="")

def emojis_to_text(text: str) -> str:
    # 😍 -> :heart_eyes: -> heart eyes
    demojized = emoji.demojize(text, delimiters=(" ", " "))
    return demojized.replace("_", " ").replace(":", "")

print("Removed :", remove_emojis(sample_text))
print("As text  :", emojis_to_text(sample_text))


Removed : OMG!!! I just watched <b>Inception</b>  at https://cinema.example.com/movie/123 & it's soooo gr8, def a masterpiece... didn't expect that twist ¯\_(ツ)_/¯ café scene was epic too. rating: 9/10 #mindblown @director_nolan
As text  : OMG!!! I just watched <b>Inception</b>  smiling face with heart-eyes  smiling face with heart-eyes  at https//cinema.example.com/movie/123 & it's soooo gr8, def a masterpiece... didn't expect that twist ¯\ (ツ) /¯ café scene was epic too. rating 9/10 #mindblown @director nolan


### 2.7 Expanding Contractions

`"didn't"` → `"did not"`, `"it's"` → `"it is"`. This matters a lot: without expansion, a tokenizer may split `"didn't"` into `["didn", "'t"]` — `"'t"` is meaningless noise, and worse, you *lose* the word "not", which is often the single most important word for sentiment analysis!


In [10]:
def expand_contractions(text: str) -> str:
    return contractions.fix(text)

print(expand_contractions("I didn't expect that, it's a masterpiece!"))


I did not expect that, it is a masterpiece!


### 2.8 Expanding Chat Slang / Abbreviations

Social text is full of informal shorthand (`u`, `ur`, `gr8`, `def`, `omg`). There's no single library that covers every slang term — the industry-standard approach is a **custom lookup dictionary** you maintain and extend as you see new terms in your data.


In [14]:
# Extend this dictionary as you discover new slang in your own datasets
SLANG_DICT = {
    # Common short forms
    "u": "you",
    "ur": "your",
    "r": "are",
    "ya": "you",
    "yr": "your",
    "n": "and",
    "b4": "before",
    "2": "to",
    "4": "for",

    # Common abbreviations
    "lol": "laughing out loud",
    "lmao": "laughing my ass off",
    "lmfao": "laughing my fucking ass off",
    "rofl": "rolling on the floor laughing",
    "omg": "oh my god",
    "omw": "on my way",
    "idk": "i do not know",
    "idc": "i do not care",
    "ik": "i know",
    "ikr": "i know right",
    "imo": "in my opinion",
    "imho": "in my humble opinion",
    "btw": "by the way",
    "brb": "be right back",
    "bbl": "be back later",
    "ttyl": "talk to you later",
    "ttys": "talk to you soon",
    "afk": "away from keyboard",

    # Agreement / reaction
    "yep": "yes",
    "yup": "yes",
    "nope": "no",
    "yeah": "yes",
    "nah": "no",
    "fr": "for real",
    "frfr": "for real for real",
    "ngl": "not gonna lie",
    "tbh": "to be honest",
    "honestly": "honestly",
    "istg": "i swear to god",
    "smh": "shaking my head",
    "fyi": "for your information",
    "afaik": "as far as i know",

    # Time / urgency
    "asap": "as soon as possible",
    "rn": "right now",
    "atm": "at the moment",
    "l8r": "later",
    "cya": "see you",
    "g2g": "got to go",
    "gtg": "got to go",

    # Thanks / requests
    "thx": "thanks",
    "thanx": "thanks",
    "ty": "thank you",
    "tysm": "thank you so much",
    "yw": "you are welcome",
    "pls": "please",
    "plz": "please",
    "np": "no problem",
    "nvm": "never mind",

    # Emotions / reactions
    "wtf": "what the fuck",
    "wth": "what the hell",
    "tf": "the fuck",
    "ffs": "for fucks sake",
    "omfg": "oh my fucking god",
    "yay": "yay",
    "ugh": "ugh",
    "aww": "aww",

    # Positive / negative
    "gr8": "great",
    "def": "definitely",
    "lit": "exciting",
    "fire": "excellent",
    "goat": "greatest of all time",
    "w": "win",
    "l": "loss",
    "sus": "suspicious",
    "mid": "average",
    "meh": "not very good",

    # Social media / internet
    "dm": "direct message",
    "pm": "private message",
    "irl": "in real life",
    "iirc": "if i remember correctly",
    "tmi": "too much information",
    "fomo": "fear of missing out",
    "yolo": "you only live once",
    "bff": "best friend forever",
    "bffl": "best friends for life",

    # Relationships / people
    "bf": "boyfriend",
    "gf": "girlfriend",
    "bday": "birthday",
    "hbd": "happy birthday",
    "fam": "family",
    "bro": "brother",
    "sis": "sister",

    # Common conversational phrases
    "wyd": "what are you doing",
    "wbu": "what about you",
    "hbu": "how about you",
    "wym": "what do you mean",
    "wdym": "what do you mean",
    "hmu": "hit me up",
    "lmk": "let me know",
    "msg": "message",
    "cmon": "come on",
    "gonna": "going to",
    "wanna": "want to",
    "gotta": "got to",
    "kinda": "kind of",
    "sorta": "sort of",

    # Your original examples
    "soooo": "so",
}

def expand_slang(text: str, slang_dict: dict = SLANG_DICT) -> str:
    words = text.split()
    expanded = [slang_dict.get(w.lower(), w) for w in words]
    return " ".join(expanded)

print(expand_slang("omg u r def gonna love this, gr8 movie tbh lol"))


oh my god you are definitely going to love this, great movie to be honest laughing out loud


### 2.9 Removing Punctuation

Punctuation rarely helps traditional (non-transformer) models. `string.punctuation` gives you every standard punctuation character to strip.

⚠️ **Task-dependent caveat:** keep punctuation for tasks like sentence-boundary detection, sarcasm/emphasis detection (`"!!!"` is a signal!), or when using transformer models — their subword tokenizers handle punctuation intelligently on their own.


In [12]:
def remove_punctuation(text: str) -> str:
    return text.translate(str.maketrans("", "", string.punctuation))

print(remove_punctuation("Wow!!! This movie... is great, isn't it?"))


Wow This movie is great isnt it


In [ ]:
# incase of keeping or removing some special punctuations
def remove_special_punctuation(text: str) -> str:
    punctuation = ".,!?;:"   # Add/remove punctuation as you need

    return text.translate(str.maketrans("", "", punctuation))


text = "Wow!!! This movie... is great, isn't it?"

print(remove_special_punctuation(text))

### 2.10 Removing Digits / Numbers

Useful when numbers add no value to your task (e.g. topic classification). But think twice — for a movie-review rating task, `"9/10"` is *extremely* informative! Always ask: **does this task need numbers?**


In [13]:
def remove_numbers(text: str) -> str:
    return re.sub(r"\d+", "", text)

print(remove_numbers("rating: 9/10, watched it in 2024"))


rating: /, watched it in 


### 2.11 Removing Extra Whitespace

After all the removals above, you're often left with double spaces, tabs, or newlines. Always run this **last** in your cleaning chain.


In [14]:
def remove_extra_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

print(repr(remove_extra_whitespace("too   much    space \n\n here")))


'too much space here'


### 2.12 Spelling Correction

The most **expensive** cleaning step (it's slow at scale), so it's used selectively — typically for user-generated content with heavy typos (reviews, support tickets, chat logs), and almost never for already-clean text like news articles or Wikipedia.


In [ ]:
spell = SpellChecker()

def correct_spelling(text: str) -> str:
    words = text.split()
    corrected = []
    for w in words:
        fix = spell.correction(w)
        corrected.append(fix if fix else w)
    return " ".join(corrected)

print(correct_spelling("this moiosve wase a masterpiece"))


this move was a masterpiece


### 2.13 Putting the Cleaning Steps Together

Here's the full chain applied in a sensible order to our messy sample. **Order matters** — e.g. expand contractions *before* removing punctuation (otherwise `"didn't"` loses its apostrophe and can't be matched), and always run whitespace cleanup *last*.


In [16]:
def clean_text(text: str,
                lowercase: bool = True,
                remove_html: bool = True,
                remove_url: bool = True,
                expand_contract: bool = True,
                convert_emoji: bool = True,
                remove_punct: bool = True,
                remove_digits: bool = False) -> str:
    if remove_html:
        text = remove_html_tags(text)
    if remove_url:
        text = remove_urls(text)
    if expand_contract:
        text = expand_contractions(text)
    if convert_emoji:
        text = emojis_to_text(text)
    text = remove_accented_chars(text)
    if lowercase:
        text = to_lowercase(text)
    if remove_punct:
        text = remove_punctuation(text)
    if remove_digits:
        text = remove_numbers(text)
    text = remove_extra_whitespace(text)
    return text

print("BEFORE:", sample_text)
print("AFTER :", clean_text(sample_text))


BEFORE: OMG!!! I just watched <b>Inception</b> 😍😍 at https://cinema.example.com/movie/123 & it's soooo gr8, def a masterpiece... didn't expect that twist ¯\_(ツ)_/¯ café scene was epic too. rating: 9/10 #mindblown @director_nolan
AFTER : omg i just watched inception smiling face with hearteyes smiling face with hearteyes at it is soooo gr8 def a masterpiece did not expect that twist tsu cafe scene was epic too rating 910 mindblown director nolan


## 3. Tokenization

**Tokenization** = splitting text into smaller units ("tokens") a model can work with. This is arguably the most important preprocessing decision, because it defines your model's entire vocabulary.

There are 3 levels of tokenization, used by different eras/types of models:

| Level | Example | Used by |
|---|---|---|
| Word-level | `"running"` → `["running"]` | Classic ML (Naive Bayes, SVM), older DL |
| Sentence-level | splits paragraphs into sentences | Summarization, sentence embeddings |
| Subword-level | `"running"` → `["run", "##ning"]` | **Modern Transformers** (BERT, GPT) — industry standard today |


### 3.1 Naive Word Tokenization — `.split()`

The simplest possible tokenizer. Works okay for quick prototyping, but fails on punctuation (`"great!"` stays glued together as one token).


In [17]:
text = clean_text(sample_text)
print(text.split())


['omg', 'i', 'just', 'watched', 'inception', 'smiling', 'face', 'with', 'hearteyes', 'smiling', 'face', 'with', 'hearteyes', 'at', 'it', 'is', 'soooo', 'gr8', 'def', 'a', 'masterpiece', 'did', 'not', 'expect', 'that', 'twist', 'tsu', 'cafe', 'scene', 'was', 'epic', 'too', 'rating', '910', 'mindblown', 'director', 'nolan']


### 3.2 Word Tokenization — NLTK

NLTK's `word_tokenize` is smarter — it correctly separates punctuation from words even when punctuation wasn't stripped.


In [18]:
tokens = word_tokenize("This movie isn't great, it's just okay.")
print(tokens)


['This', 'movie', 'is', "n't", 'great', ',', 'it', "'s", 'just', 'okay', '.']


### 3.3 Sentence Tokenization — NLTK

Splits a paragraph into individual sentences — used before per-sentence tasks (e.g. summarization, sentence-level sentiment).


In [19]:
paragraph = "This movie was great. I loved the acting! Would I watch it again? Absolutely."
sentences = sent_tokenize(paragraph)
for i, s in enumerate(sentences, 1):
    print(f"{i}. {s}")


1. This movie was great.
2. I loved the acting!
3. Would I watch it again?
4. Absolutely.


### 3.4 Word Tokenization — spaCy (industry standard for classical pipelines)

spaCy is faster, more accurate on messy real-world text, and gives you rich token objects (each token already knows its lemma, POS tag, and more — see Sections 5 & 6). This is what most production NLP pipelines use instead of NLTK today.


In [20]:
doc = nlp("This movie isn't great, it's just okay.")
spacy_tokens = [token.text for token in doc]
print(spacy_tokens)


['This', 'movie', 'is', "n't", 'great', ',', 'it', "'s", 'just', 'okay', '.']


### 3.5 Subword Tokenization — the Transformer/Industry Standard

Word-level tokenization has a big weakness: any word not seen during training becomes an unknown `<UNK>` token, and the vocabulary size explodes for large corpora. **Subword tokenization** (Byte-Pair Encoding / WordPiece) solves this by breaking rare words into common sub-pieces, so *every* word can be represented — even ones never seen before.

This is exactly what powers BERT, GPT, and virtually every modern LLM. We use HuggingFace's `tokenizers`/`transformers` library, the industry-standard tool for this.

```bash
pip install transformers
```


In [21]:
# This cell needs internet access to fetch the pretrained tokenizer files.
# It is included for completeness / future reference — skip it if offline.
try:
    from transformers import AutoTokenizer
    bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    subword_tokens = bert_tokenizer.tokenize("Preprocessing unbelievably transforms tokenization.")
    print(subword_tokens)
    # Notice how rare/long words like "unbelievably" get split into known sub-pieces
    # instead of becoming a single unknown token.
except Exception as e:
    print("Skipped (needs internet / model download):", e)


Skipped (needs internet / model download): No module named 'transformers'


## 4. Stopword Removal

**Stopwords** are extremely common, low-information words: `"the"`, `"is"`, `"a"`, `"and"`... They appear in almost every sentence regardless of topic, so removing them lets your model focus on words that actually carry meaning.

⚠️ **Critical caveat:** words like `"not"`, `"no"`, `"never"` are stopwords in NLTK's default list — but removing them **destroys sentiment meaning**! `"not good"` → `"good"` completely flips the meaning. Always customize your stopword list for sentiment/negation-sensitive tasks. Also, **skip this step entirely for transformers** — they use surrounding context and don't benefit from stopword removal (it can even hurt performance).


In [25]:
stop_words = set(stopwords.words("english"))
print(f"Total NLTK English stopwords: {len(stop_words)}")
print(list(stop_words)[:15])
print(stop_words)

Total NLTK English stopwords: 198
['all', 'them', 'in', "you've", 'mightn', 'my', 'very', 'that', 'which', 'ours', 'are', 'up', 'hers', 'just', 'how']
{'all', 'them', 'in', "you've", 'mightn', 'my', 'very', 'that', 'which', 'ours', 'are', 'up', 'hers', 'just', 'how', 'yours', 'himself', 'out', 'against', "he's", "mustn't", 'off', "shouldn't", 'these', "wouldn't", 'here', 'those', 'he', "doesn't", "they'll", 'not', 'your', 'the', 'll', 'isn', 'to', 'doing', 'being', 'as', 'into', 'below', 're', "we'd", 'an', 'his', 'there', 'will', 'between', 'of', "she's", 'whom', 'do', "wasn't", "weren't", 'should', 'from', "didn't", "he'll", 'm', 'about', 'and', 'both', "they'd", 'its', 'why', "i'm", "won't", 'so', 'her', "hasn't", 'shouldn', 'doesn', 'other', 'can', "hadn't", 'don', "we'll", 'or', "needn't", "i'd", 'hadn', 'needn', "she'll", 'this', 'it', 'what', 'our', 'where', "he'd", "mightn't", 'above', "they're", 'by', 'once', 'i', 'herself', 'their', "it'd", 'aren', 'me', 'then', 'until', 'aga

In [26]:
def remove_stopwords(text: str, stop_words: set = stop_words) -> str:
    tokens = word_tokenize(text)
    filtered = [w for w in tokens if w.lower() not in stop_words]
    return " ".join(filtered)

print(remove_stopwords("this is a great movie and i really loved it"))


great movie really loved


In [24]:
# Negation-safe stopword list — keep the words that carry sentiment polarity
NEGATION_WORDS = {"not", "no", "nor", "never", "n't", "none", "nothing"}
safe_stopwords = stop_words - NEGATION_WORDS

print(remove_stopwords("this movie was not good and i did not enjoy it", safe_stopwords))


movie not good not enjoy


## 5. Stemming vs Lemmatization

Both reduce words to a base form so `"running"`, `"runs"`, and `"ran"` are treated as related. They work very differently:

| | Stemming | Lemmatization |
|---|---|---|
| Method | Chops off suffixes with rules (fast, crude) | Uses vocabulary + grammar rules (slower, accurate) |
| Output | May not be a real word (`"studies"` → `"studi"`) | Always a real dictionary word (`"studies"` → `"study"`) |
| Speed | Very fast | Slower |
| When to use | Search engines, large-scale indexing where speed > precision | Sentiment analysis, chatbots, anything needing accuracy |


### 5.1 Stemming — Porter & Snowball

Porter is the classic, oldest algorithm. Snowball is an improved, more consistent version of it — generally preferred today when stemming is used at all.


In [25]:
porter = PorterStemmer()
snowball = SnowballStemmer("english")

words_to_test = ["running", "studies", "flies", "happily", "generously", "better"]

for w in words_to_test:
    print(f"{w:12s} -> porter: {porter.stem(w):10s} | snowball: {snowball.stem(w)}")


running      -> porter: run        | snowball: run
studies      -> porter: studi      | snowball: studi
flies        -> porter: fli        | snowball: fli
happily      -> porter: happili    | snowball: happili
generously   -> porter: gener      | snowball: generous
better       -> porter: better     | snowball: better


### 5.2 Lemmatization — NLTK WordNet

Notice `"better"` correctly lemmatizes to `"good"` when we tell it the word is an adjective (`pos="a"`) — something stemming can *never* do.


In [ ]:
lemmatizer = WordNetLemmatizer()

print(lemmatizer.lemmatize("running", pos="v"))   # -> run & parts of speech (pos):verb
print(lemmatizer.lemmatize("studies", pos="n"))   # -> study & parts of speech (pos):noun
print(lemmatizer.lemmatize("better", pos="a"))    # -> good & parts of speech (pos):adjective


run
study
good


### 5.3 Lemmatization — spaCy (industry standard: context-aware, no manual POS needed)

This is the big advantage of spaCy: it automatically figures out each word's grammatical role from context, so you never have to manually pass a POS tag like with NLTK.


In [27]:
doc = nlp("The striped bats were hanging on their feet and eating better than the studies suggested.")
lemmas = [token.lemma_ for token in doc]
print(" ".join(lemmas))


the striped bat be hang on their foot and eat well than the study suggest .


## 6. POS Tagging & Named Entity Recognition (NER)

These aren't strictly "cleaning" steps, but they're extremely common **preprocessing signals** used to build smarter pipelines — e.g. keeping only nouns/adjectives for topic modeling, or masking out person names before feeding text to a model (privacy).


### 6.1 Part-of-Speech (POS) Tagging

Labels every word with its grammatical role (noun, verb, adjective...).


In [28]:
tagged = pos_tag(word_tokenize("The quick brown fox jumps over the lazy dog"))
print(tagged)


[('The', 'DT'), ('quick', 'JJ'), ('brown', 'NN'), ('fox', 'NN'), ('jumps', 'VBZ'), ('over', 'IN'), ('the', 'DT'), ('lazy', 'JJ'), ('dog', 'NN')]


### 6.2 Named Entity Recognition (NER) — spaCy

Finds real-world entities: people, organizations, dates, locations. Useful for anonymizing data or extracting structured info from free text.


In [29]:
doc = nlp("Christopher Nolan directed Inception, which was released in 2010 by Warner Bros.")
for ent in doc.ents:
    print(f"{ent.text:20s} -> {ent.label_}")


Christopher Nolan    -> PERSON
2010                 -> DATE
Warner Bros.         -> ORG


## 7. The Reusable Pipeline — `TextPreprocessor`

This is the piece you'll actually copy-paste into future projects. It's built like a **scikit-learn transformer** (`fit` / `transform`), so it drops straight into an sklearn `Pipeline` alongside `TfidfVectorizer`, a classifier, etc.

Every step is a toggle — flip only what your task needs (see the decision table in Section 10 for guidance on *what* to toggle for common tasks).


In [30]:
from sklearn.base import BaseEstimator, TransformerMixin


class TextPreprocessor(BaseEstimator, TransformerMixin):
    """
    A configurable, sklearn-compatible text cleaning pipeline.

    Example
    -------
    >>> pre = TextPreprocessor(remove_stopwords_flag=True, lemmatize=True)
    >>> pre.transform(["I didn't like this movie's <br/> pacing at all!"])
    ['not like movie pace']
    """

    def __init__(self,
                 lowercase: bool = True,
                 remove_html: bool = True,
                 remove_url: bool = True,
                 expand_contractions_flag: bool = True,
                 convert_emoji: bool = True,
                 remove_punct: bool = True,
                 remove_digits: bool = False,
                 remove_stopwords_flag: bool = True,
                 keep_negations: bool = True,
                 lemmatize: bool = True,
                 stem: bool = False):
        self.lowercase = lowercase
        self.remove_html = remove_html
        self.remove_url = remove_url
        self.expand_contractions_flag = expand_contractions_flag
        self.convert_emoji = convert_emoji
        self.remove_punct = remove_punct
        self.remove_digits = remove_digits
        self.remove_stopwords_flag = remove_stopwords_flag
        self.keep_negations = keep_negations
        self.lemmatize = lemmatize
        self.stem = stem

        self._lemmatizer = WordNetLemmatizer()
        self._stemmer = SnowballStemmer("english")
        base_stopwords = set(stopwords.words("english"))
        self._stopwords = (base_stopwords - NEGATION_WORDS) if keep_negations else base_stopwords

    def fit(self, X, y=None):
        return self  # nothing to learn -- this is a stateless cleaner

    def _clean_one(self, text: str) -> str:
        if self.remove_html:
            text = remove_html_tags(text)
        if self.remove_url:
            text = remove_urls(text)
        if self.expand_contractions_flag:
            text = expand_contractions(text)
        if self.convert_emoji:
            text = emojis_to_text(text)
        text = remove_accented_chars(text)
        if self.lowercase:
            text = to_lowercase(text)
        if self.remove_punct:
            text = remove_punctuation(text)
        if self.remove_digits:
            text = remove_numbers(text)
        text = remove_extra_whitespace(text)

        tokens = word_tokenize(text)

        if self.remove_stopwords_flag:
            tokens = [t for t in tokens if t.lower() not in self._stopwords]

        if self.lemmatize:
            tokens = [self._lemmatizer.lemmatize(t) for t in tokens]
        elif self.stem:
            tokens = [self._stemmer.stem(t) for t in tokens]

        return " ".join(tokens)

    def transform(self, X, y=None):
        return [self._clean_one(str(text)) for text in X]


# Quick smoke test
pre = TextPreprocessor()
print(pre.transform([sample_text]))


['omg watched inception smiling face hearteyes smiling face hearteyes soooo gr8 def masterpiece not expect twist tsu cafe scene epic rating 910 mindblown director nolan']


## 8. Applying the Pipeline to Real Data

Now let's run our reusable class on the actual IMDB reviews `df` from Section 1.2 (or your full 50K-row dataset from Notebook 01 — same code, zero changes needed).


In [31]:
pre = TextPreprocessor(remove_stopwords_flag=True, lemmatize=True)
df["clean_review"] = pre.transform(df["review"])

pd.set_option("display.max_colwidth", 100)
df[["review", "clean_review"]].head()


,review,clean_review
0,One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked.,one reviewer mentioned watching 1 oz episode hooked
1,A wonderful little production. <br /><br />The filming technique is very unassuming- very old-ti...,wonderful little production filming technique unassuming oldtimebbc fashion
2,"I thought this was a wonderful way to spend time on a too hot summer weekend, sitting in the air...",thought wonderful way spend time hot summer weekend sitting air conditioned theater
3,Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his...,basically family little boy jake think zombie closet parent fighting time
4,"Petter Mattei's ""Love in the Time of Money"" is a visually stunning film to watch. Mr. Mattei off...",petter matteis love time money visually stunning film watch mr mattei offer u vivid portrait hum...


## 9. From Text to Numbers — Vectorization

Models don't understand words — they understand numbers. Vectorization converts your cleaned tokens into numeric vectors. This is technically the *next* stage after preprocessing (feature engineering), but it's included here because it's the natural checkpoint: "preprocessing is done, now what?"


### 9.1 Bag of Words (BoW)

Counts how many times each word appears. Simple, but ignores word importance and order.


In [32]:
bow_vectorizer = CountVectorizer(max_features=20)
bow_matrix = bow_vectorizer.fit_transform(df["clean_review"])

bow_df = pd.DataFrame(bow_matrix.toarray(), columns=bow_vectorizer.get_feature_names_out())
bow_df


,1010,2024,absolutely,air,bad,basically,boy,brilliant,closet,conditioned,family,fashion,fighting,film,filming,little,not,time,watch,wonderful
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,0,1
2,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1
3,0,0,0,0,0,1,1,0,1,0,1,0,1,0,0,1,0,1,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,1,0
5,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0
6,1,1,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0


### 9.2 TF-IDF (Term Frequency – Inverse Document Frequency)

Improves on BoW by **downweighting** words that appear in almost every document (less informative) and **upweighting** words that are rare/distinctive. This is the industry-standard baseline vectorizer for classical ML text pipelines.


In [33]:
tfidf_vectorizer = TfidfVectorizer(max_features=20)
tfidf_matrix = tfidf_vectorizer.fit_transform(df["clean_review"])

tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out())
tfidf_df.round(2)


,1010,2024,absolutely,air,bad,basically,boy,brilliant,closet,conditioned,family,fashion,fighting,film,filming,little,not,time,watch,wonderful
0,0.00,0.00,0.00,0.00,0.00,0.0,0.0,0.00,0.0,0.00,0.0,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00
1,0.00,0.00,0.00,0.00,0.00,0.0,0.0,0.00,0.0,0.00,0.0,0.54,0.0,0.00,0.54,0.45,0.00,0.00,0.00,0.45
2,0.00,0.00,0.00,0.56,0.00,0.0,0.0,0.00,0.0,0.56,0.0,0.00,0.0,0.00,0.00,0.00,0.00,0.40,0.00,0.46
3,0.00,0.00,0.00,0.00,0.00,0.4,0.4,0.00,0.4,0.00,0.4,0.00,0.4,0.00,0.00,0.33,0.00,0.29,0.00,0.00
4,0.00,0.00,0.00,0.00,0.00,0.0,0.0,0.00,0.0,0.00,0.0,0.00,0.0,0.61,0.00,0.00,0.00,0.52,0.61,0.00
5,0.00,0.00,0.00,0.00,0.45,0.0,0.0,0.00,0.0,0.00,0.0,0.00,0.0,0.00,0.00,0.00,0.89,0.00,0.00,0.00
6,0.43,0.43,0.43,0.00,0.00,0.0,0.0,0.43,0.0,0.00,0.0,0.00,0.0,0.36,0.00,0.00,0.00,0.00,0.36,0.00


> 🔮 **Looking ahead:** for deep learning / transformer models, you'd replace BoW/TF-IDF with **word embeddings** (Word2Vec, GloVe) or a pretrained transformer's own embeddings — and skip most of Sections 2–5 entirely (see the table in Section 10). That's a topic for its own notebook.


## 10. Industry Best Practices & Common Pitfalls

### 10.1 The single most important lesson

> **Preprocessing is not one-size-fits-all. It is entirely task- and model-dependent.**

Here's the decision table big tech NLP teams actually use:

| Step | Classical ML (BoW/TF-IDF) | Sentiment Analysis | Transformers (BERT/GPT) | Search / Information Retrieval |
|---|---|---|---|---|
| Lowercasing | ✅ Yes | ✅ Yes | ⚠️ Only if using an *uncased* model | ✅ Yes |
| Remove punctuation | ✅ Yes | ⚠️ Keep `!`, `?` (emphasis signal) | ❌ No — tokenizer handles it | ✅ Yes |
| Remove stopwords | ✅ Yes | ⚠️ Keep negations (`not`, `never`) | ❌ No — hurts performance | ✅ Yes |
| Stemming/Lemmatization | ✅ Yes | ✅ Lemmatize (more accurate) | ❌ No — subword tokenizer replaces this | ✅ Yes |
| Emoji handling | Remove | Convert to text (signal!) | Usually keep as-is | Remove |
| Tokenization | Word-level | Word-level | **Subword (BPE/WordPiece)** | Word-level |

### 10.2 Order of operations matters

A wrong order silently breaks things:
- Expand contractions **before** removing punctuation (`"didn't"` needs its apostrophe to be recognized).
- Remove HTML/URLs **before** lowercasing or tokenizing (a broken-up `<br/>` tag scatters junk tokens everywhere).
- Always run whitespace cleanup **last** (every previous step leaves gaps behind).

### 10.3 Common pitfalls

- **Removing "not" as a stopword** in a sentiment task — silently flips your labels' meaning.
- **Over-cleaning for transformers** — stripping punctuation/stopwords for BERT/GPT models actively *reduces* accuracy, since these models were pretrained on natural, "dirty" text.
- **Spell-correcting everything at scale** — it's slow (one API/dictionary call per word); only apply it to genuinely noisy sources (user reviews, support chats), never to already-clean corpora (news, Wikipedia).
- **Fitting a vectorizer (TF-IDF/BoW) on the full dataset before train/test split** — this leaks test-set vocabulary statistics into training. Always `fit` only on the training set.
- **Ignoring domain-specific stopwords** — e.g. in a movie review dataset, the word `"movie"` itself appears everywhere and carries no discriminative signal; consider adding domain words to your stopword set.

### 10.4 What changed with the rise of Transformers

Pre-2018 (classical ML / early deep learning) pipelines used almost everything in Sections 2–5. Modern transformer pipelines (BERT, GPT, and what powers most of today's production NLP) intentionally do **less**:

1. Minimal cleaning (maybe strip HTML/URLs, keep everything else).
2. **No** stopword removal.
3. **No** stemming/lemmatization.
4. Subword tokenization is *built into* the model's own tokenizer — you don't build this yourself.

This is why, in industry today, you'll see two very different-looking preprocessing pipelines depending on which kind of model a team is shipping.


## 11. Quick-Reference Cheat Sheet

Every function from this notebook, in one place, ready to copy-paste into a new project.

```python
import re, string, unicodedata
import emoji, contractions
from unidecode import unidecode
from spellchecker import SpellChecker
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer
import spacy
nlp = spacy.load("en_core_web_sm")

for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)

# ---- Cleaning ----
def remove_html_tags(text):     return re.sub(r"<.*?>", "", text)
def remove_urls(text):          return re.sub(r"https?://\S+|www\.\S+", "", text)
def remove_mentions(text):      return re.sub(r"@\w+", "", text)
def strip_hashtag_symbol(text): return re.sub(r"#(\w+)", r"\1", text)
def remove_accented_chars(text): return unidecode(text)
def remove_emojis(text):        return emoji.replace_emoji(text, replace="")
def emojis_to_text(text):
    return emoji.demojize(text, delimiters=(" ", " ")).replace("_", " ").replace(":", "")
def expand_contractions(text):  return contractions.fix(text)
def remove_punctuation(text):   return text.translate(str.maketrans("", "", string.punctuation))
def remove_numbers(text):       return re.sub(r"\d+", "", text)
def remove_extra_whitespace(t): return re.sub(r"\s+", " ", t).strip()

SLANG_DICT = {"u": "you", "ur": "your", "r": "are", "gr8": "great", "def": "definitely",
              "omg": "oh my god", "idk": "i do not know", "btw": "by the way",
              "imo": "in my opinion", "lol": "laughing out loud", "brb": "be right back"}
def expand_slang(text, slang_dict=SLANG_DICT):
    return " ".join(slang_dict.get(w.lower(), w) for w in text.split())

spell = SpellChecker()
def correct_spelling(text):
    return " ".join(spell.correction(w) or w for w in text.split())

# ---- Stopwords (negation-safe) ----
NEGATION_WORDS = {"not", "no", "nor", "never", "n't", "none", "nothing"}
STOPWORDS = set(stopwords.words("english")) - NEGATION_WORDS
def remove_stopwords(text, stop_words=STOPWORDS):
    return " ".join(w for w in word_tokenize(text) if w.lower() not in stop_words)

# ---- Stemming / Lemmatization ----
lemmatizer = WordNetLemmatizer()
def lemmatize_spacy(text):
    return " ".join(token.lemma_ for token in nlp(text))

# ---- The all-in-one reusable class: TextPreprocessor (Section 7) ----
# Copy the full class definition from Section 7 — sklearn-compatible,
# every step toggle-able via constructor arguments:
#
#   pre = TextPreprocessor(remove_stopwords_flag=True, lemmatize=True)
#   df["clean_review"] = pre.transform(df["review"])
```

### One-line decision guide

- **Building a quick sklearn baseline (BoW/TF-IDF)?** → Use the full `TextPreprocessor` with all flags on.
- **Sentiment analysis?** → `keep_negations=True`, prefer `lemmatize=True` over `stem`, keep emojis as text.
- **Fine-tuning BERT/GPT/any transformer?** → Skip almost everything — just strip HTML/URLs, pass raw-ish text straight to the model's own tokenizer.
- **Building a search index?** → Full cleaning + stemming (speed matters more than perfect accuracy at scale).

---
**End of notebook.** Everything above is copy-paste ready — good luck with the project! 🚀
